# 03b · 公开数据进货：官方数据直接进云

## 这一章在整个项目里的位置

```
00 ✅ → 01 ✅ → 02 ✅ → 03 数据进货（大门 + 全蛋白组） → [03b 公开数据进货] → 04–09 Databricks → 10–12
```

03 号建好了大门（验货员、收货单），收进了第一批货：全蛋白组预测。这一章把 QAFI 还要用的**公开数据**一次进齐：ClinVar、UniProt、ProteinGym、AlphaFold（v4 和 v6）。

## 背景：为什么要这样进数据

**以前的做法**：从各个网站手动下载到自己电脑，自己整理，然后在代码里直接读。时间一长，就说不清"用的是哪个版本、从哪下的、整理时改了什么"。一个现成的例子：QAFI 的代码里，AlphaFold 有 7 处用 v4、5 处用 v6，两个版本混着用。

**公司里的做法**：
1. 一张**数据源清单**，写清楚每份数据从哪来、哪个版本、多久更新
2. 云上的程序按清单，**直接从官方网站搬进 raw 仓库**，不经过任何人的电脑
3. 每份数据都有版本文件夹和收货单
4. 之后所有的整理都从 raw 开始、写成代码（04、05 号），任何结果都能追溯到某一份原始数据

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 比喻 |
|---|---|---|---|
| 1 | 列一张数据源清单 | 配置文件 `config/sources.toml` | 进货单 |
| 2 | 写一个送货员 | 第二个 Lambda | 送货车 |
| 3 | 分块搬运，到了核对 | HTTP 分段下载 + S3 分块上传 | 大件拆开运，到了点数 |
| 4 | 用图纸建出来 | Terraform | 按图施工 |
| 5 | 一次进齐，看收货单 | | 签收 |

**花费**：一共约 11 GB，存储每月约 0.3 美元。Lambda 每月有 40 万 GB·秒的免费额度，这一章只用掉一两千。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 AWS，读出 raw 桶的名字。**不用看代码。**

In [ ]:
import json, os, subprocess
from pathlib import Path
import boto3
from qafi_platform import ingest
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
aws = boto3.Session(region_name="eu-central-1")
s3, lam = aws.client("s3"), aws.client("lambda")
RAW = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names", shell=True, capture_output=True, text=True, check=True).stdout)["raw"]
run("aws sts get-caller-identity --query Arn --output text")
print("raw 桶:", RAW)

> 如果报错说登录过期：在终端运行 `aws login --profile qafi`，然后重新运行这一格。

---

## 第 1 步 · 数据源清单

### 在干什么
把"要进哪些数据"写成一张清单：[`config/sources.toml`](../../config/sources.toml)。

### 背景
数据从哪来，是整个平台最基础的记录。写在某个人的脑子里或者散在代码各处，迟早说不清。写成一张清单，就有了**唯一的答案**。

### 用了什么技术，为什么
清单是一个配置文件（TOML 格式，一种人能直接读的格式）。每份数据一段：

| 数据 | 版本 | 官方多久更新 | QAFI 里干什么 |
|---|---|---|---|
| `clinvar` | 2026-10 | 每月 | 评估用的临床标签（0.914 那个数） |
| `uniprot-human` | 2026_03 | 每 8 周 | 蛋白编号、序列、基因名；04 号修 GeneID 那一列要用 |
| `proteingym` | v1.3 | 不定期 | DMS 实验数据：训练和评估 |
| `alphafold-human` | v4 | 很少 | 结构（QAFI 代码大多用这一版） |
| `alphafold-human` | v6 | 很少 | 结构（最新版） |

**版本写死**：每份数据进 raw 的位置是 `datasets/<名字>/<版本>/`。官方出了新版本，就改清单里的版本号，新数据进新文件夹，旧的不动。

**故意不放的**：
- **PDB**：AWS 上本来就有一份公开的、每周更新的副本。我们只用到其中几个结构，到 07 号用到哪个直接读哪个，不复制整个库
- **HGMD**：有授权限制，不进云、不进代码库
- **全蛋白组预测**：不是公开数据，03 号已经从你的电脑传了

### 公司里什么情况怎么做
- **加一份新数据**：在清单里加一段，提交，走 Pull Request 审查。代码不用改
- **官方出了新版本**：改版本号。06 号会把这件事变成自动的：按 `refresh` 定时检查、自动进货

### 面试怎么说
> "Ingestion is config-driven: a version-controlled source list says what we ingest, from where, at which version and how often. Adding a dataset is a one-entry change, not new code."

---

## 第 2 步 · 送货员：第二个 Lambda

### 在干什么
写一个 Lambda：给它一个官方下载地址和 raw 里的位置，它就把文件搬过去。

### 背景
从你电脑下载、再从你电脑上传：慢（家里网络的上传速度很低），占着你的电脑，而且以后没法自动重复。云上的程序直接从官方网站搬到 S3：带宽大，几分钟搬完 5 GB，谁的电脑都不用开。

### 用了什么技术，为什么
还是 **Lambda**：一次调用搬一个文件，7 个文件就同时叫 7 个送货员。搬完的文件落进 `datasets/`，**03 号的门铃照样响，验货员照样检查、开收货单**。大门只有一个，不管货是谁送来的。

送货员还会在文件上记下"**从哪来**"：官方地址和官方的发布日期。验货员把它抄进收货单。以后任何人看到一个文件，都能查到它的出处。

代码在 [`src/qafi_platform/raw_fetcher.py`](../../src/qafi_platform/raw_fetcher.py)，有 5 个单元测试：用一个本机的小网站假扮官方下载站，测试分块搬运、不支持分段的网站、出错时的清理。

### 公司里什么情况怎么做
- **单个文件几十 GB，或者要跑超过 15 分钟**：Lambda 不够了，改用容器任务（ECS / Fargate、AWS Batch），道理一样
- **整个文件系统级别的大搬家**：用 AWS DataSync

### 面试怎么说
> "Public datasets are copied publisher-to-S3 by a Lambda, never through a laptop. Each object records its source URL and release date, and lands in the same raw zone, so the same validator and receipts apply."

---

## 第 3 步 · 分块搬运，到了核对

### 在干什么
5 GB 的文件，怎么在 15 分钟内搬完，并且确认没搬错。

### 用了什么技术，为什么
**分段下载**：官方网站允许"只给我第 X 到第 Y 个字节"（HTTP Range）。送货员把文件切成 64 MB 一段，**8 段同时下载、同时上传**，就是 03 号"分块上传"的思路，下载这一头也分块。哪一段断了，只重下那一段。

**到了核对**：官方大多没有提供指纹，所以核对两件事：
1. **大小**：S3 里文件的字节数，必须和官方声明的一模一样。防止"只搬了一半"
2. **每一块的指纹**：每块带着 CRC32 指纹，S3 收到后自己再算一遍，对不上就拒收。防止"搬错了"。整个文件的指纹 S3 也会算好存下来，写在收货单上

**失败了怎么办**：任何一块失败，整个上传作废，不会留下半个文件。送货员**不自动重试**：重试会悄悄再下载 5 GB，所以失败就在日志里留下错误，由人决定重新运行。

### 公司里什么情况怎么做
- **官方提供了指纹**（比如 UniProt 的 `RELEASE.metalink` 文件），就再加一层：和官方指纹比对

### 面试怎么说
> "Large files are copied with parallel HTTP range requests into an S3 multipart upload with per-part CRC32 checksums; the stored size must match the publisher's, and a failed copy is aborted rather than left half-written."

---

## 第 4 步 · 用图纸建出来

### 在干什么
送货员写在图纸 [`infra/dev/fetch.tf`](../../infra/dev/fetch.tf) 里，plan → apply 建出来。

### 用了什么技术，为什么
这次 plan 的结果是 **`6 to add, 1 to change`**：
- **6 个新资源**：送货员本身，它的工牌和权限（只能往 `datasets/` 里写），日志本，以及"失败不自动重试"的设置
- **1 个修改**：`aws_lambda_function.raw_validator will be updated in-place`。验货员的代码改了两处：认识了 `.tar` 文件（AlphaFold 的结构包是 tar 格式，它的文件签名不在开头，在第 257 个字节），以及把"从哪来"抄进收货单。**改了代码 → plan 显示 update in-place → apply 就部署了新代码**，03 号说过的就是这个

### 执行

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")

### 结果怎么看
- plan：6 行 `will be created`，1 行 `will be updated in-place`，最后 `Plan: 6 to add, 1 to change, 0 to destroy.`
- apply：最后 `Apply complete! Resources: 6 added, 1 changed, 0 destroyed.`

> 如果 plan 显示 `No changes`：说明已经建好了（比如别人已经 apply 过），直接进下一步。

---

## 第 5 步 · 一次进齐

### 在干什么
按清单，同时叫 7 个送货员，等所有收货单出来。

### 执行
读清单：

In [ ]:
sources = ingest.load_sources(ROOT / "config" / "sources.toml")
for s in sources: print(s["key"])

进货。实测：小文件几秒，两个 AlphaFold 各约 2 分半，全部 2.5 分钟：

In [ ]:
receipts = ingest.fetch_sources(lam, s3, RAW, sources)

### 结果怎么看
- 先是 7 行 `sent ...`：7 个送货员出发了
- 然后每份一行 `accepted ...` 加大小。AlphaFold 两个各约 5.1 GB

> 第一次进货已经在 2026-10-07 做完了：7 个全部 `accepted`，大小和官网声明的完全一致。所以你现在运行这一格，会看到 7 行 `skipped`，原因见下面。

**已经进过的不会再搬**：同一个版本、同一个官方地址，已经有 `accepted` 收货单的，显示 `skipped (already in raw)`，不会再复制一遍。这叫**幂等**（idempotent）：同样的操作重复运行，结果不变、不产生重复数据。数据管道必须这样，因为定时任务、失败重跑都会让同一个操作执行好几次。

看一张收货单，注意 `source`，它记着这个文件从哪来：

In [ ]:
print(json.dumps(receipts[-2], indent=2))

看送货员的工作记录：每个文件搬了多少块、用了几秒：

In [ ]:
run("aws logs tail /aws/lambda/qafi-dev-raw-fetcher --since 3h --format short --filter-pattern '{ $.level = \"INFO\" }'")

### 结果怎么看
每个文件一行。第一次进货时的实际记录，比如 `copied datasets/alphafold-human/v4/UP000005640_9606_HUMAN_v4.tar (5113649152 bytes, 77 parts) in 138.3 s`：5 GB 分成 77 块，2 分多钟。小文件是 `single stream`，一两秒。

### 公司里什么情况怎么做
- **有一份没有收货单**：先看送货员的日志（上面这一格去掉 `--filter-pattern` 那一段，能看到 ERROR），修好原因后重新运行这一格。已经进来的会被跳过，只搬缺的那份
- **想强制重新搬**：`ingest.fetch_sources(lam, s3, RAW, sources, force=True)`。新的会成为一个新版本，旧的还在（版本控制）

### 面试怎么说
> "Ingestion is idempotent: a file already accepted from the same source is skipped, so scheduled runs and retries never duplicate data."

---

## 第 6 步 · 以后谁来更新

现在是手动进货。**06 号**会把它变成自动的：定时任务按清单里的 `refresh`（比如 ClinVar 每月）检查有没有新版本，有就更新版本号、进货、等收货单，再触发 04、05 号重建表。进货和整理按顺序由同一个调度系统管，这就是 06 号的内容。

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 03b config-driven public data ingestion") && git push -q')

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 清单 | `config/sources.toml` | 数据从哪来，只有一个答案；加数据只改清单 |
| 2 送货员 | 第二个 Lambda | 官方数据直接进云，不经过个人电脑 |
| 3 分块 + 核对 | HTTP Range + S3 分块上传 | 大文件搬得快；大小和指纹都对得上 |
| 4 图纸 | Terraform | 改代码 → update in-place → 部署 |
| 5 进齐 | 幂等的进货 | 重复运行不重复搬 |

**一句话**：QAFI 需要的原始数据，现在都在 raw 里，每一份都有版本、出处和收货单。从下一章开始，所有整理都从这里出发。

## 面试题

1. **How do you ingest public reference data such as ClinVar or AlphaFold?**
   > "From a version-controlled source list: a Lambda copies each file from the publisher straight into a versioned raw prefix, recording the source URL and release date; the raw validator checks it and writes a receipt. A scheduler re-runs it when the publisher releases."

2. **Why pin dataset versions?**
   > "Reproducibility: a model or result must trace back to the exact inputs. New releases go into new version folders; switching a model to them is a deliberate, gated decision."

3. **What does idempotent mean for a pipeline, and why does it matter?**
   > "Running it twice gives the same result as once. Retries, backfills and schedules re-run steps all the time; without idempotency you get duplicates or corrupted state."

4. **When is Lambda the wrong tool for ingestion?**
   > "When a single task needs more than 15 minutes or 10 GB of memory, or a persistent environment: then a container task on ECS/Fargate or AWS Batch, or DataSync for bulk transfers."

**下一章 04 · Databricks**：把 raw 里的数据放进 Databricks，变成能用 SQL 查询的表，并开始清洗。